[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-Crew/transport-networks-lab/blob/main/Notebooks/Practica_guiada2_red_ciclovias.ipynb)


#  Grafos y redes de transporte

![informacion](https://github.com/PyMap/AUPY/blob/master/imagenes/grafos_introduccion.png?raw=1)

##  Parte 2. Teoría de grafos con `NetworkX`
### Cuestiones prácticas

#### Estructurando la red EcoBici en la Ciudad de Buenos Aires

Como cualquier sistema de transporte, la red de bicicletas puede entenderse como una estructura sencilla de nodos conectados por ejes o arcos.

Normalmente, en una red de transporte los nodos se encuentran representados por las estaciones. Por su parte, los ejes pueden tomar la forma de las vías que los conectan - ya sean las calles o carriles que delimitan el recorrido en cuestión -, o incluso, los viajes realizados entre nodos.


![caption](https://github.com/PyMap/AUPY/blob/master/imagenes/estructura_red.png?raw=1)

De tal manera, vamos a reconocer tres dimensiones en las que deberemos ir tomando algunas decisiones para poder representar la red pública de bicicletas como un grafo:

**1. Componentes de la red:**

Dado que la red EcoBici se encuentra organizada en un conjunto delimitado de estaciones, vamos a considerar a estas últimas como los **nodos** de nuestro grafo y a las rutas transitadas, como la conexiones entre los mismos.

En ese sentido, no debemos perder de vista que los viajes realizados entre nodos responden a una necesidad de desplazamiento. Y por lo tanto, pueden asumir distintos sentidos. Esto nos lleva a plantear una segunda cuestión ...
    
**2. Tipo de red:**

Los viajes realizados en la red pueden tener un mismo punto de partida y de llegada, o contrariamente, ir de uno a otro.

Por lo tanto, deberemos definir si nuestro grafo será de tipo dirigido o no, si soportará self-loops y ejes paralelos.

    
**3. Conectividad dentro la red:**

Por último, vamos a evaluar si existe algún patrón de movilidad identificable a partir de la intensidad de las conexiones.

In [ ]:
from google.colab import drive
drive.mount('/drive/')

## Sección 1: Armando nuestro grafo

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import networkx as nx
!pip install plotly_express
import plotly_express as px

In [ ]:
#@title Cargamos datasets de recorridos de bicicletas

import pandas as pd
# https://data.buenosaires.gob.ar/dataset/bicicletas-publicas
# Usamos un diccionario para almacenar los dataframes de distintos anios
df = {}
for i in range(15,20):
    path = f'/drive/MyDrive/Técnicas y Análisis de datos del Transporte/Data/bici{str(i)}_cdn.csv'
    df[str(i)] = pd.read_csv(path)

In [ ]:
df.keys()

In [ ]:
# Previsualizamos
df['18'].head()

In [ ]:
#@title Cargamos las estaciones y verificamos su localización
import geopandas as gpd
from shapely import wkt

# Shape de estaciones descargados de DataBA
estaciones_wkt = pd.read_csv('/drive/MyDrive/Técnicas y Análisis de datos del Transporte/Data/estaciones_ecobici.csv', index_col=[0])
proj = '+proj=tmerc +lat_0=-34.6297166 +lon_0=-58.4627 +k=0.9999980000000001 +x_0=100000 +y_0=100000 +ellps=intl +units=m +no_defs'

# load geom from wkt
def from_wkt(df, wkt_column, crs=4326):

    df["geometry"]= df[wkt_column].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=crs)

    gdf.drop(columns=wkt_column, inplace=True)
    return gdf

estaciones = from_wkt(estaciones_wkt, 'str_geom').to_crs(proj)

In [ ]:
# Previsualizamos
estaciones.plot();

In [ ]:
print('Total estaciones: {}'.format(len(estaciones)))

### 1.2. Nodos, ejes y sentido de las conexiones

Antes de seguir avanzando, debemos tener en cuenta que el sistema público de bicicletas fue diseñado para funcionar sobre una red de ciclovías.  Se puede ir de una estación a otra o biceversa, como así también, salir y llegar a la misma estación siempre utilizando una única vía.

Asimismo, todas las ciclovías tienen una dirección de ida y otra de vuelta. No hay casos en los que haya un sentido y casos donde haya otro, como puede suceder con la red de de tránsito vehicular.

Por lo tanto, estamos frente a un caso de caminos bidireccionales. **Entonces si el orden importa, ¿es nuestra red un grafo dirigido o no dirigido?**

![caption](https://github.com/PyMap/AUPY/blob/master/imagenes/dirigido_nodirigido.png?raw=1)

Como vemos en la ilustración, en un grafo dirigido sólo hay un arco entre (por ejemplo) A y B. Se puede ir de A a B pero no de B a A. Suponiendo que estas fuesen nuestras estaciones, la forma que adoptarían en un grafo dirigido sería `A -> B`.

Ahora bien, como dijimos anteriormente, un ciclista puede ir de A a B o de B a A dependiendo hacia a dónde necesite desplazarse. En nuestro grafo, los viajes tienen direccionalidad y las estaciones podrían conectarse en ambos sentidos (se puede ir de A a B y de B a A). Esto nos deja dos posibilidades.

**a.** Si planteamos un grafo no dirigido, las estaciones deberían tomar la forma `A - B`.

**b.** Si planteamos un grafo dirigido, las estaciones deberían tomar la forma `A <-> B`

Ambas alternativas son posibles, probemos alguna de ellas. Para eso, tomemos un año al azar. Digamos, 2018.

In [ ]:
len(df['18'])

In [ ]:
# Antes de armar nuestro grafo revisemos que el dataset sea consistente
print('Estaciones de origen sin informacion:{}'.format(len(df['18'][df['18'].bici_estacion_origen.isna()])))
print('Estaciones de destino sin informacion:{}'.format(len(df['18'][df['18'].bici_estacion_destino.isna()])))

In [ ]:
# no contamos con información para algunas estaciones de destino
df['18'][df['18'].bici_estacion_destino.isna()]

In [ ]:
# hay varias estrategias de imputación, una podría ser suponer que estos movimientos son self-loops
df['18'].bici_estacion_destino.fillna(df['18'].bici_estacion_origen)

In [ ]:
def graphmaker(df, origen, destino, atributo, constructor=None):
    '''
    Devuelve un grafo compuesto por estaciones(nodos) y recorridos(vinculos) para cada dataframe.
    ...
    Argumentos:

    df = dataframe de recorridos
    origen = Serie del dataframe referenciando la estacion de origen del grafo
    destino = Serie del dataframe referenciando la estacion de destino del grafo
    '''

    if len(df[df[origen].isna()])>0:
        print('Imputando estaciones de origen')
        df[origen] = df[origen].fillna(df[destino])

    if len(df[df[destino].isna()])>0:
        print('Imputando estaciones de destino')
        df[destino] = df[destino].fillna(df[origen])

    if len(df[df[atributo].isna()])>0:
        print('Imputando promedios para el atributo')
        df[atributo] = df[atributo].fillna(df[atributo].mean())

    bicigrafo = nx.from_pandas_edgelist(df=df,
                                        source=origen,target=destino,
                                        edge_attr=atributo,
                                        create_using=constructor)

    return bicigrafo

In [ ]:
# Probemos la funcion para crear un grafo a partir de un dataframe cualquiera
b18 = graphmaker(df=df['18'],origen='bici_nombre_estacion_origen',
                 destino='bici_nombre_estacion_destino', atributo='bici_tiempo_uso')

In [ ]:
# Ahora veamos qué nos devuelve. Como no especificamos el constructor, un grafo no dirigido.
print(type(b18))

In [ ]:
print("Se crea un Grafo de %r nodos con %r arcos" % (b18.number_of_nodes(), b18.number_of_edges()))

Siguiendo esta lógica, obtenemos un grafo de 199 nodos con un poco menos de 20.000 arcos.
Hasta acá, estructuramos una red de conexiones a partir de un dataframe de pandas que representa los vínculos entre pares de estaciones. Aunque, ¿cómo podemos estar seguros de que esta representación es la correcta?

Primero veamos cuántas combinaciones posibles hay en el universo de estaciones disponible. Así sabremos, cuántos son los pares únicos de nodos que podrían estar conectandose ...

In [ ]:
# agrupamos las estaciones por origen y destino (el dataframe se modifico inplace en nuestra funcion!)
agrupado = df['18'].groupby(['bici_nombre_estacion_origen',
                             'bici_nombre_estacion_destino']).size().reset_index(name='viajes')

In [ ]:
agrupado.head()

El groupby toma una estacion y la contrasta contra todas las restantes computando así la cantidad de casos que hay para cada par único de orígenes y destinos.

In [ ]:
# de minima, sabemos que las combinaciones posibles entre estaciones de origen y destino son
len(agrupado)

Ahora, vemos que las combinaciones posibles son casi el doble de los arcos que obtuvimos inicialmente. Entonces, ¿estamos perdiendo vínculos si trabajamos bajo la forma de un  `grafo no dirigido`?

Primero, tratemos de entender cuál sería el universo de conexiones posibles en función del tipo de grafo y después tratemos de contrastar las conexiones que efectivamente se producen.

En los grafos no dirigidos no importa el orden de la conexión porque los nodos pueden vincularse en ambos sentidos. Esto hace que la cantidad máxima de ejes entre nodos se piense como una combinación de "k" elementos tomados de  un grupo de "n" elementos.

![caption](https://github.com/PyMap/AUPY/blob/master/imagenes/formulas.png?raw=1)

Sin embargo, nuestro problema responde más bien a una permutación que a una combinación. Principalmente, porque el orden de las conexiones sí importa. Cada par de estaciones es único, e ir de A a B o de B a A no es lo mismo para nosotros. Entonces, no deberíamos utilizar en el denominador `k!` para conocer el número máximo de pares `origen - destino`.

In [ ]:
# Primero armamos una funcion para devolver el factorial de un nro.
def factorial(n):
    if n == 0:
        return 1
    else:
        fac = 1
        for i in range(1, n + 1):
            fac *= i
        return fac

En un grafo no dirigido no se tiene en cuenta el orden. `A-B | B-A` es lo mismo (se divide por 2 para contar la combinación de pares sólo una vez!)

In [ ]:
# Si nos diera lo mismo ir de A a B o de B a A, la cantidad de ejes máximos rondaría los ...
Lmax = factorial(200)/((factorial(200-2))*(factorial(2)))
Lmax

En otras palabras, algo bastante similar al total de ejes que contamos en nuestro grafo no dirigido. Esto quiere decir que, si el orden no importa, estamos dejando de contar conexiones que son únicas por su sentido. Tomemos el grafo de 2018 y revisemos cómo se conectan un par de estaciones al azar.

In [ ]:
conexiones = pd.DataFrame(b18.edges(data=True))
conexiones.columns = ['origen','destino','atributo']
conexiones.loc[((conexiones['origen']=='15 de Noviembre') | (conexiones['origen']=='25 de Mayo')) &
              ((conexiones['destino']=='15 de Noviembre') | (conexiones['destino']=='25 de Mayo'))]

Bien, hay autoconexiones. Aunque curiosamente, en el grafo no se computó el par `15 de Noviembre - 25 de Mayo` pero sí `25 de Mayo - 15 de Noviembre`. Esto quiere decir que, al momento de instanciarse se consideró solamente un par de estaciones, sin importar su sentido.

In [ ]:
# Corroborémoslo, ese par no está en el grafo!
conexiones.loc[(conexiones['origen']=='15 de Noviembre') & (conexiones['destino']=='25 de Mayo')]

... pero sí en el dataset de viajes. Constatemos ahora en nuestra base que el recorrido `15 de Noviembre -  25 de Mayo` efectivamente existe.

In [ ]:
agrupado.columns = ['origen','destino','viajes']

agrupado.loc[((agrupado['origen']=='15 de Noviembre') | (agrupado['origen']=='25 de Mayo')) &
             ((agrupado['destino']=='15 de Noviembre') | (agrupado['destino']=='25 de Mayo'))]

La clase `graph` que utilizamos para construir nuestro grafo considera los self loops pero no contempla ejes múltiples. Es por eso que los que son pares únicos de nodos no se procesan como tales. Ir de 15 de Noviembre a 25 de Mayo o vicebersa es lo mismo y se computa el primer valor disponible en el orden de indexación cuando aplicamos el método `nx.from_pandas_edgelist`.

Ahora que sabemos que nuestros recorridos son singulares y que no nos da lo mismo ir de un lado a otro, podemos decir que la `direccionalidad` es un atributo que debemos respetar para poder crear un grafo que represente correctamente el comportamiento de la red EcoBici. Pero, ¿cómo lo conseguimos?

Primero evaluemos qué cantidad de conexiones posibles habríamos de esperar si respetamos el sentido de los ejes. Es decir, si se tiene en cuenta que ir de A-B | B-A no es lo mismo.

In [ ]:
# calculamos el límite máximo de pares origen - destino en un contexto dirigido
Lmax = factorial(200)/factorial(200-2)
Lmax

Ya no dividimos por 2 porque nuestras conexiones son singulares. Así obtenemos un valor mucho más parecido al total que obtuvimos en nuestro dataframe de nodos agrupados.

Es decir, si consideramos singularmente ir de A a B y de B a A a lo largo de toda la red, el total de combinaciones posibles entre pares únicos de estaciones debería rondar en ese valor. Y decimos debería porque es posible que no todos los nodos estén conectados de manera adyacente.

Nuestro valor máximo de pares únicos conectados será `39800`. Ahora, podremos tener más o menos ejes según haya adyacencia entre nodos, como así también si existen relaciones paralelas o múltiples. Esta es la otra cuestión de relevancia!

Pero antes veamos qué sucede trabajando con un `grafo dirigido`.

In [ ]:
# creamos un grafo dirigido
b18_di = graphmaker(df=df['18'],origen='bici_nombre_estacion_origen',
                    destino='bici_nombre_estacion_destino', atributo='bici_tiempo_uso',
                    constructor=nx.DiGraph)

In [ ]:
print("Se crea un Grafo de %r nodos con %r arcos" % (b18_di.number_of_nodes(), b18_di.number_of_edges()))

![caption](https://github.com/PyMap/AUPY/blob/master/imagenes/movimientos.png?raw=1)

En este caso, el número de arcos se asemeja mucho más al total de pares únicos que obtuvimos agrupando nodos por origen y destino. Ahora bien, ¿solamente respetando la direccionalidad es que garantizamos la totalidad de ejes o conexiones presentes en la red?

En realidad no. Si bien es cierto que nos acercamos bastante al valor máximo que estipulamos en la permutación, lo que alcanzamos con el grafo dirigido es un valor cercano al total de conexiones que se dan *al menos una vez* entre pares adyacentes de nodos. Más concretamente, no debemos perder de vista que la clase `DiGraph` de NetworkX tampoco soporta ejes mútltiples. Por ende, si hay un viaje que se efectúa más de una vez entre un mismo par de nodos, el mismo no será computado.

Por lo tanto, sabemos que un grafo dirigido nos devolverá al menos una conexión entre A -> B | B -> A si esta existe. Pero si hubiera más de un viaje en la misma direccionalidad no se va a contar dos veces.

Por eso, la `direccionalidad` no es lo único que tenemos que tener en cuenta para instanciar nuestro grafo. Sino también la posibilidad de soportar `ejes paralelos o múltiples`.

In [ ]:
# probemos ahora con un multigrafo no dirigido!
b18_multi = graphmaker(df=df['18'],origen='bici_nombre_estacion_origen',
                       destino='bici_nombre_estacion_destino', atributo='bici_tiempo_uso',
                       constructor=nx.MultiGraph)

In [ ]:
print("Se crea un Grafo de %r nodos con %r arcos" % (b18_multi.number_of_nodes(), b18_multi.number_of_edges()))

Contemplando ejes múltiples, vemos que la cantidad de conexiones sube considerablemente. Pero no perdamos de vista que es un grafo no dirigido y que la relación es `A - B`. Es decir, no hay direccionalidad. No se respeta la singularidad de la conexión como con la clase `DiGraph`, pero al soportar ejes múltiples, se cuenta ir de A a B y vicebersa tantas veces como aparezca el par de nodos en el dataframe.

Está mal? No necesariamente. Es una forma de representar la red, la cual implica suponer que las estaciones están conectadas por ciclovías bidireccionales. Esto, sin tener en cuenta el sentido de la conexión.

In [ ]:
# tratemos por último con un multigrafo dirigido
b18_multidi = graphmaker(df=df['18'],origen='bici_nombre_estacion_origen',
                         destino='bici_nombre_estacion_destino', atributo='bici_tiempo_uso',
                         constructor=nx.MultiDiGraph)

In [ ]:
print("Se crea un Grafo de %r nodos con %r arcos" % (b18_multidi.number_of_nodes(), b18_multidi.number_of_edges()))

Tanto dirigido como no dirigido, pero contemplando ejes múltiples obtenemos la misma cantidad de conexiones. Hagamos un último testeo para ver en qué se diferenciarían ambas alternativas.

Para eso, vamos a volver sobre el mismo para de estaciones con las que trabajamos antes y ver las diferencias de frecuencias que hay en el número de ejes para cada clase de grafo.

In [ ]:
def evalua_resultados(grafo, origen='15 de Noviembre', destino='25 de Mayo'):

    print("Ejes totales entre pares:")
    print("*************************")

    print("O-D:{}".format(grafo.number_of_edges(u=origen,v=destino)))
    print("D-O:{}".format(grafo.number_of_edges(u=destino,v=origen)))
    print("O-O:{}".format(grafo.number_of_edges(u=origen,v=origen)))
    print("D-D:{}".format(grafo.number_of_edges(u=destino,v=destino)))

    print("Ejes totales:{}".format(grafo.number_of_edges()))

In [ ]:
# grafo no dirigido
evalua_resultados(b18)

In [ ]:
# grafo dirigido
evalua_resultados(b18_di)

In [ ]:
# multigrafo
evalua_resultados(b18_multi)

In [ ]:
# multigrafo dirigido
evalua_resultados(b18_multidi)

In [ ]:
agrupado.viajes.sum()

Estudiando más de cerca lo que sucede entre las estaciones `25 de Mayo` y `15 de Noviembre` podemos decir que la principal diferencia es cómo se contemplan los ejes. Mientras en un multigrafo no dirigido se tratan como relaciones recíprocas, en uno dirigido se diferencian según el sentido.

Lo importante, es que, si instanciamos nuestro grafo con eje paralelos vamos a poder respetar el total de viajes de todos los usuarios de la red. Ahora, la manera como trabajemos el comportamiento de la misma dependerá tanto de su topología como de lo que estemos buscando representar.

In [ ]:
# creemos el resto de los grafos, los vamos a necesitar!
b19_multidi = graphmaker(df=df['19'],origen='bici_nombre_estacion_origen',
                         destino='bici_nombre_estacion_destino', atributo='bici_tiempo_uso',
                         constructor=nx.MultiDiGraph)

b15_multidi = graphmaker(df=df['15'],origen='bici_nombre_estacion_origen',
                         destino='bici_nombre_estacion_destino', atributo='bici_tiempo_uso',
                         constructor=nx.MultiDiGraph)

b16_multidi = graphmaker(df=df['16'],origen='bici_nombre_estacion_origen',
                         destino='bici_nombre_estacion_destino', atributo='bici_tiempo_uso',
                         constructor=nx.MultiDiGraph)

b17_multidi = graphmaker(df=df['17'],origen='bici_nombre_estacion_origen',
                         destino='bici_nombre_estacion_destino', atributo='bici_tiempo_uso',
                         constructor=nx.MultiDiGraph)

## Sección 2: representando la red

Avancemos un poco más. Ahora que sabemos que no existe una única forma de representar nuestra red tratemos de ver cómo esto se relaciona con su representación gráfica.

Si bien es cierto que nuestra representación debe ajustarse lo más posible a la realidad (respetar ejes paralelos, direccionalidad, etc) no debemos perder de vista que los grafos, como estructura de datos, es un tipo de representación costosa en memoria.

Por lo tanto, a la hora de representar gráficamente una red también debemos pensar en qué es lo que estamos buscando reflejar. En nuestro caso, la red EcoBici, es un sistema eminentemente de caminos adyacentes (un usuario toma una bicicleta en una estación de origen, devolviéndola en otra de destino).

Por ende, tratar la red como un grafo no dirigido únicamente para ver estas relaciones puede que no esté tan mal. Veamos cómo resulta...

In [ ]:
# rearmamos graphmaker para poder incorporar las coordenadas de las estaciones
def graphmaker(df, origen, destino, atributo, constructor=None):
    '''
    Devuelve un grafo compuesto por estaciones(nodos) y recorridos(vinculos) para cada dataframe.
    ...
    Argumentos:

    df = dataframe de recorridos
    origen = Serie del dataframe referenciando la estacion de origen del grafo
    destino = Serie del dataframe referenciando la estacion de destino del grafo
    '''

    if len(df[df[origen].isna()])>0:
        print('Imputando estaciones de origen')
        df[origen] = df[origen].fillna(df[destino])

    if len(df[df[destino].isna()])>0:
        print('Imputando estaciones de destino')
        df[destino] = df[destino].fillna(df[origen])

    if len(df[df[atributo].isna()])>0:
        print('Imputando promedios para el atributo')
        df[atributo] = df[atributo].fillna(df[atributo].mean())

    estaciones_en_viajes = pd.concat([df[origen].astype(int),
                                      df[destino].astype(int)]).unique()

    # nos quedamos con las estaciones para las que tenemos coordenadas
    estaciones_en_df = estaciones[estaciones.NRO_EST.isin(estaciones_en_viajes)].copy()
    estaciones_en_df['coordenadas'] = list(zip(estaciones_en_df.geometry.y,
                                               estaciones_en_df.geometry.x))
    coordenadas = dict(zip(estaciones_en_df.NRO_EST, estaciones_en_df.coordenadas))

    # filtramos coordenadas existentes en nuestras estaciones de origen y destino
    df = df[df[origen].isin(pd.Series(coordenadas.keys()))].copy()
    df = df[df[destino].isin(pd.Series(coordenadas.keys()))].copy()

    bicigrafo = nx.from_pandas_edgelist(df=df,
                                        source=origen,target=destino,
                                        edge_attr=atributo,
                                        create_using=constructor)

    # agregamos las coordenadas al grafo
    for k,v in coordenadas.items():
        bicigrafo.nodes[k]['coord'] = v

    return bicigrafo

In [ ]:
# instanciamos nuestro grafo con el nuevo atributo de coordenadas
G19 = graphmaker(df=df['19'],
                 origen='bici_estacion_origen',
                 destino='bici_estacion_destino',
                 atributo='bici_tiempo_uso')

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# y lo graficamos!
fig, ax = plt.subplots(figsize=(17,10))

pos = nx.get_node_attributes(G19,'coord')

nx.draw_networkx_edges(G=G19, pos=pos, width=0.01)

nx.draw_networkx_nodes(G=G19,
                       pos=pos,
                       node_size=10,
                       node_color='#00203FFF')

ax.set_axis_off()
ax.set_title('Red EcoBici - 2019');

## **Ejercitación**

Ahora que vimos distintas formas de representar nuestra red y cómo ubicarla en el espacio ...

1. Utilizar la función `graphmaker` e instanciar un grafo para cada uno de los años disponibles.
   Pista: nosotros queremos ver relaciones adyacentes, tener cuidado con el tipo de clase que vamos a utilizar!
2. Armar una función que consuma cada grafo, lo plotee en ejes diferentes y asigne un color por año a los nodos.

In [ ]:
# Desarrollar ejercicio


In [ ]:
#@title **Solución**
G15 = graphmaker(df=df['15'],
                 origen='bici_estacion_origen',
                 destino='bici_estacion_destino',
                 atributo='bici_tiempo_uso')

G16 = graphmaker(df=df['16'],
                 origen='bici_estacion_origen',
                 destino='bici_estacion_destino',
                 atributo='bici_tiempo_uso')

G17 = graphmaker(df=df['17'],
                 origen='bici_estacion_origen',
                 destino='bici_estacion_destino',
                 atributo='bici_tiempo_uso')

G18 = graphmaker(df=df['18'],
                 origen='bici_estacion_origen',
                 destino='bici_estacion_destino',
                 atributo='bici_tiempo_uso')

def plotea_grafos(G1, G2, color1, color2, año1, año2):
    # y lo graficamos!
    fig = plt.figure(figsize=(20,11))
    ax1 = fig.add_subplot(2,2,1)
    ax2 = fig.add_subplot(2,2,2)

    pos1 = nx.get_node_attributes(G1,'coord')
    pos2 = nx.get_node_attributes(G2,'coord')

    nx.draw_networkx_edges(G=G1, pos=pos1, width=0.01, ax=ax1)
    nx.draw_networkx_nodes(G=G1,pos=pos1, node_size=10,node_color=color1, ax=ax1)

    nx.draw_networkx_edges(G=G2, pos=pos2, width=0.01, ax=ax2)
    nx.draw_networkx_nodes(G=G2,pos=pos2, node_size=10,node_color=color2, ax=ax2)


    ax1.set_axis_off()
    ax1.set_title('Red EcoBici - {}'.format(año1))

    ax2.set_axis_off()
    ax2.set_title('Red EcoBici - {}'.format(año2));

In [ ]:
# utilizamos la funcion de ploteo
plotea_grafos(G1=G15, G2=G16, color1='#00A4CCFF', color2='#F95700FF', año1='2015', año2='2016')

In [ ]:
plotea_grafos(G1=G17, G2=G18, color1='#DAA03DFF', color2='#39FF14', año1='2017', año2='2018')

## Sección 3: métricas y atributos de la red

Ahora que vimos la implicancia de cada tipo de representación, exploremos algunas métricas y veamos qué poder decir sobre la evolución de la red de bicicletas.

### 3.1. Cantidad de nodos y ejes

In [ ]:
# Cantidad de nodos(estaciones) y ejes(recorridos) de cada dataset
n15, e15 = b15_multidi.number_of_nodes(), b15_multidi.number_of_edges()
n16, e16 = b16_multidi.number_of_nodes(), b16_multidi.number_of_edges()
n17, e17 = b17_multidi.number_of_nodes(), b17_multidi.number_of_edges()
n18, e18 = b18_multidi.number_of_nodes(), b18_multidi.number_of_edges()
n19, e19 = b19_multidi.number_of_nodes(), b19_multidi.number_of_edges()

In [ ]:
# Evolucion de la cantidad de estaciones y recorridos realizados entre 2015 y 2019
evolucion = pd.DataFrame({'Año':['2015','2016','2017','2018','2019'],
                          'Nodos':[n15,n16,n17,n18,n19],
                          'Ejes':[e15,e16,e17,e18,e19]})

In [ ]:
evolucion

In [ ]:
# El grado promedio por año para nuestros grafos dirigidos
evolucion['Grado_promedio'] = round(evolucion['Ejes']/evolucion['Nodos'],1)

In [ ]:
evolucion

In [ ]:
# Vemos como evolucionaron la cantidad de nodos y edges
fig, ax = plt.subplots(figsize=(12, 5))

# Plot "Nodos"
ax.plot(evolucion["Año"], evolucion["Nodos"], label="Nodos", marker='o')
ax.set_ylabel("Estaciones (nodos)")
ax.set_xlabel("Año")
ax.set_xticks(evolucion["Año"])
ax.set_xticklabels(evolucion["Año"])

# Crear eje y-secundario
ax2 = ax.twinx()
ax2.plot(evolucion["Año"], evolucion["Ejes"], label="Ejes", color="r", marker='o')
ax2.set_ylabel("Recorridos (edges)")

# Title and grid
plt.title('Evolución de la cantidad de nodos y ejes en la red de bicicletas. CABA-2015/2019', y=1.05)
plt.grid()

# Combine legends
fig.legend(loc='upper left', bbox_to_anchor=(0.15,0.85))

# Show plot
plt.show();

In [ ]:
# Dejamos afuera a 2019 (no incluye todos los meses, solamente hasta febrero)
evolucion_f = evolucion.iloc[:4].copy()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(evolucion_f["Año"], evolucion_f["Nodos"], label="Nodos", marker='o')
ax.set_ylabel("Estaciones (nodos)")
ax.set_xlabel("Año")
ax.set_xticks(evolucion_f["Año"])
ax.set_xticklabels(evolucion_f["Año"])

ax2 = ax.twinx()
ax2.plot(evolucion_f["Año"], evolucion_f["Ejes"], label="Ejes", color="r", marker='o')
ax2.set_ylabel("Recorridos (edges)")

plt.title('Evolución de la cantidad de nodos y ejes en la red de bicicletas. CABA-2015/2018', y=1.05)
plt.grid()

fig.legend(loc='upper left', bbox_to_anchor=(0.15,0.85))

plt.show()

In [ ]:
print('Entre 2016 y 2017, los nodos de la red EcoBici creciieron un %s%%' % round((n17/n16-1)*100,2))

In [ ]:
print('Por su parte, la cantidad de recorridos (ejes) crecio un %s%%'% round((e17/e16-1)*100,2))

En primera instancia, algo que podemos ir viendo sobre nuestra red es que el volumen de nodos y conexiones entre los mismos es creciente a lo largo de los años. Ahora, enfoquemonos en un año en concreto y evaluemos cuáles fueron los nodos que más conexiones establecieron.

Para eso, debemos recordar que **el grado de un nodo se define como la cantidad de de ejes adyacentes al mismo** ...

In [ ]:
# La conexión entre una estación y otra es el viaje realizado
print(f"El grado promedio de la red para 2018 fue de {int((b18_multidi.number_of_edges())/b18_multidi.number_of_nodes())} ejes o conexiones")

In [ ]:
# esta sentencia de pyplot define un estilo global
plt.style.use('ggplot')

# esta es una función auxiliar para plotear barras!
def plotear_barras_horizontales(df, bar_color, titulo):
    df.plot.barh(color=bar_color, figsize=(12,6))

    legend = plt.legend(frameon = 1)
    frame = legend.get_frame()

    frame.set_facecolor('white')
    frame.set_edgecolor('black')
    frame.set_linewidth(1)

    plt.title(titulo)
    plt.ylabel(None)
    plt.xlabel(df.columns[0])

    plt.show()

# ... que utilizaremos en nuestra función de centralidad!
def centralidad(grafo, medida, top, show_worst, titulo, bar_color):
    if medida == 'node_degree':
        node_degree = pd.DataFrame(list(grafo.degree(grafo.nodes())))
        node_degree.columns = ['estacion','node degree']
        node_degree.index = node_degree.estacion
        node_degree.drop(columns=['estacion'], inplace=True)
        node_degree = node_degree.sort_values(by='node degree', ascending=show_worst)
        plotear_barras_horizontales(df=node_degree.head(top), bar_color=bar_color, titulo=titulo)

    if medida == 'degree_centrality':
        degree_centrality = nx.degree_centrality(G=grafo)
        degree_centrality = pd.DataFrame(degree_centrality.values(), index=degree_centrality.keys())
        degree_centrality.columns = ['degree_centrality']
        degree_centrality = degree_centrality.sort_values(by='degree_centrality', ascending=show_worst)
        plotear_barras_horizontales(df=degree_centrality.head(top), bar_color=bar_color, titulo=titulo)

In [ ]:
centralidad(grafo=b18_multidi, medida='node_degree', show_worst = False,
            top=5, titulo='Centralidad de la red - 2018', bar_color='#616247FF')

In [ ]:
# Y si vemos los nodos con mas baja conectividad?
centralidad(grafo=b18, medida='degree_centrality', show_worst=True,
            top=5, titulo='Centralidad de la red - 2018', bar_color='#8d8d8d')

Por lo pronto, durante 2018, la estación más utilizada fue `Facultad de Medicina`. Ahora bien, ¿cómo fue el comportamiento de las conexiones de este nodo con el resto?

### 3.1. Direccionalidad de los nodos centrales

In [ ]:
def evalua_conexiones_del_nodo(grafo, estacion):
    nodos = list(grafo.nodes())
    conexiones_dirigidas_u = {}
    conexiones_dirigidas_v = {}

    # registramos todas las conexiones dirigidas de la estacion
    for n in nodos:
        conexiones_dirigidas_u[n] = grafo.number_of_edges(u=estacion,v=n)
        conexiones_dirigidas_v[n] = grafo.number_of_edges(u=n,v=estacion)

    return conexiones_dirigidas_u, conexiones_dirigidas_v

In [ ]:
# Creamos todas las conexiones en sentido u/v del nodo Facultad de Medicina
estacion_u, estacion_v = evalua_conexiones_del_nodo(grafo=b18_multidi, estacion='Facultad de Medicina')

In [ ]:
# y armamos un dataframe para guardar las conexiones en ambas direcciones
estacion_dfu = pd.DataFrame(estacion_u.values(), estacion_u.keys())
estacion_dfu.columns = ['conexiones_u']

estacion_dfv = pd.DataFrame(estacion_v.values(), estacion_v.keys())
estacion_dfv.columns = ['conexiones_v']

estacion_df =estacion_dfu.join(estacion_dfv).copy()
estacion_df['conexiones'] = estacion_df.conexiones_u + estacion_df.conexiones_v

In [ ]:
# sabemos que el resultado es correcto, porque
estacion_df.conexiones.sum()

In [ ]:
# es igual al grado del nodo
b18_multidi.degree['Facultad de Medicina']

**Interpretación:**

* se registran 90 viajes entre `Facultad de Medicina`(como estación de origen) y `Parque Lezama` (como estación de destino)
* se registran 94 viajes entre `Parque Lezama`(como estación de origen) y `Facultad de Medicina` (como estación de destino)

In [ ]:
estacion_df.head()

In [ ]:
# Por lo tanto sabemos que en esta estacion, la mayoria de las conexiones fueron self loops
estacion_df.sort_values(by='conexiones', ascending=False).head(1)

Algo adicional que vemos con este análisis es que, en la estación más usada de la red durante 2018, los usuarios retiraron y volvieron a depositar su bicicleta en la misma estación.

### 3.2.  Conexiones direccionales: in y out degree

Si recuerdan, dijimos que en un grafo dirigido el grado de un nodo es el resultado de las conexiones entrantes y salientes. Veamos para el mismo año con el que venimos trabajando cómo es este comportamiento y su relación con el grado del nodo en las estaciones de mayor centralidad.

In [ ]:
# Creamos una funcion para devolver un dataframe con el grado del nodo, y las conexiones entrantes/salientes
def ndg(grafo):
    '''
    Devuelve un df con cada nodo, su degree, in_degree y out_degree
    '''
    ndg = grafo.degree()
    idg = grafo.in_degree()
    odg = grafo.out_degree()

    estacion = [k for k,v in ndg]
    in_degree = [v for k,v in idg]
    out_degree = [v for k,v in odg]
    node_degree = [v for k,v in ndg]

    df = pd.DataFrame(estacion).reset_index()
    df.drop(columns='index', inplace=True)
    df.columns = ['estacion']
    df['in_degree'],df['out_degree'],df['node_deg'] = in_degree, out_degree, node_degree

    return df

In [ ]:
# Instanciamos la funcion ...
ndg18 = ndg(b18_multidi)

In [ ]:
ndg18[ndg18.estacion=='Facultad de Medicina']

In [ ]:
def plotea_node_degree_estaciones(df):
    plt.figure(figsize=(15,6))
    plt.plot('estacion', 'node_deg', data=df, marker='o', markerfacecolor='blue',
             markersize=7, color='skyblue', linewidth=3, label='node_degree')
    plt.plot('estacion', 'in_degree', data=df, marker='', color='red', linewidth=1)
    plt.plot('estacion', 'out_degree', data=df, marker='', color='olive', linewidth=2, linestyle='dashed')

    plt.title('Node, In y Out degree - Estaciones Ecobici 2018',y=1.02)
    plt.xticks(np.arange(len(df)),df['estacion'] , fontsize=9, rotation=45)

    legend = plt.legend(frameon = 1)
    frame = legend.get_frame()
    frame.set_facecolor('white')
    frame.set_edgecolor('black')
    frame.set_linewidth(1)
    plt.grid(axis='y', alpha=0.5);

In [ ]:
# y graficamos!
plotea_node_degree_estaciones(df=ndg18.sort_values(by='node_deg', ascending=False).head(10))

### 3.3. Distribucion de grados y matrices de adyacencia

Algo de lo que vimos en la evolución de nodos y ejes es que, la cantidad de ambos era incremental. Ahora evaluemos cómo es su distribución para un mismo año. Esto también nos dará una noción sobre cómo están conectados los nodos y cuan completas son las conexiones adyacentes del grafo ...

In [ ]:
plt.figure(figsize=(15,5))

plt.subplot(1,1,1)
nd18 = pd.Series([b18_multidi.degree(n) for n in b18_multidi.nodes()])
nd18.plot.hist(grid=True, bins=20, rwidth=0.9,
                   color='#616247FF')

plt.title('Distribución del grado de los nodos - EcoBici 2018');

In [ ]:
plt.figure(figsize=(15,5))

plt.subplot(1,1,1)

# Retomar digresion: la mayoria de los nodos alcanzan la totalidad de conexiones posibles dentro de la red
dc18 = pd.Series(nx.degree_centrality(b18).values())

# Otra forma de calcularla
#dc18 = pd.Series([b18.degree[n]/(b18_multidi.number_of_nodes()-1) for n in b18_multidi.nodes()])

dc18.plot.hist(grid=True, bins=20, rwidth=0.9,
                   color='#8d8d8d')

plt.title('Distribución del grado de centralidad de los nodos - EcoBici 2018');

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
cax = ax.matshow(nx.to_numpy_array(G18), cmap='YlGnBu')

# eliminamos la grilla
ax.set_xticks([])
ax.set_yticks([])

plt.title("Matriz de adyacencia - 2018");

In [ ]:
def plot_matrices(matrices, titles, figsize=(20, 10)):
    fig, axes = plt.subplots(1, len(matrices), figsize=figsize)

    for ax, matrix, title in zip(axes, matrices, titles):
        ax.matshow(matrix, cmap=plt.get_cmap('YlGnBu'))
        ax.set_axis_off()
        ax.set_title(title)

    plt.show()

matrices = [nx.to_numpy_array(G15), nx.to_numpy_array(G16), nx.to_numpy_array(G17), nx.to_numpy_array(G18)]
titles = ['EcoBici 2015', 'EcoBici 2016', 'EcoBici 2017', 'EcoBici 2018']

plot_matrices(matrices, titles)

Tal y como dijimos antes, la red EcoBici muestra una evolución creciente de conexiones adyacentes. Esto no habla más que de una red que se va completando a lo largo del tiempo. Con más usuarios y conexiones bidireccionales entre nodos.

## Sección 4: patrones de uso

In [ ]:
#@title 4.1. Matrices horarias

import seaborn as sns

# Creamos las tablas pivot
anios = ['15', '16', '17', '18']
pivot_tables = {}

for anio in anios:
    pivot_tables[f'pivot{anio}'] = pd.pivot_table(df[anio],
                                                  values="bici_sexo",
                                                  index=pd.to_datetime(df[anio].bici_Fecha_hora_retiro).dt.hour,
                                                  columns=pd.to_datetime(df[anio].bici_Fecha_hora_retiro).dt.weekday,
                                                  aggfunc='count',
                                                  fill_value=0)

pivot15 = pivot_tables['pivot15']
pivot16 = pivot_tables['pivot16']
pivot17 = pivot_tables['pivot17']
pivot18 = pivot_tables['pivot18']

fig, axes = plt.subplots(1, 4, figsize=(24, 10))

pivot_tables = {
    '2015': pivot15,
    '2016': pivot16,
    '2017': pivot17,
    '2018': pivot18
}

# Y ploteamos
dias_semana = ['Lu', 'Ma', 'Mi', 'Ju', 'Vi', 'Sa', 'Do']

for ax, (year, pivot) in zip(axes, pivot_tables.items()):
    sns.heatmap(pivot, square=True, cmap=sns.diverging_palette(180, 10, as_cmap=True), linewidths=.1, ax=ax)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=360)
    labels = [item.get_text() + ' hs' for item in ax.get_yticklabels()]
    ax.set_yticklabels(labels, rotation=0)
    ax.set_ylabel("")
    ax.set_xticklabels(dias_semana, rotation=360)
    ax.set_xlabel("")
    ax.set_title(year, y=1.01)

fig.suptitle('Retiro de bicicletas (en cantidad de personas) por hora. CABA-2015/2018', fontsize=16)
plt.show()

In [ ]:
#@title 4.2. Armamos algunas funcionalidades para hacer recortes temporales de nuestra base de usuarios
# 1. Definimos una función para convertir todas las referencias temporales a datetime
def atributos_temporales(dict_input,key):
    """
    Agrega columnas con diferentes atributos temporales
    en un dataframe de pandas.
    ...
    Argumentos:
     dict_input (dict): diccionario de dataframes con viajes por año.
     key (str): año

    Devuelve:
      pandas.dataframe : df con series de mes,fecha,hora y dia.
    """

    df = dict_input[key].copy()
    df['mes'] = pd.to_datetime(df.bici_Fecha_hora_retiro).dt.month
    df['fecha'] = pd.to_datetime(df.bici_Fecha_hora_retiro).dt.date.astype('datetime64[ns]')
    df['hora'] = pd.to_datetime(df.bici_Fecha_hora_retiro).dt.hour
    df['dia_semana'] = pd.to_datetime(df.bici_Fecha_hora_retiro).dt.weekday
    df['nombre_dia_semana'] = df['dia_semana'].replace({0:'lunes',
                                                        1:'martes',
                                                        2:'miercoles',
                                                        3:'jueves',
                                                        4:'viernes',
                                                        5:'sabado',
                                                        6:'domingo'})

    return df


# 2. Definimos una función para filtrar nuestro dataset en fechas y horarios específicos
def construye_filtros(df, filtro):
    """
    Filtra un dataframe de pandas a partir de
    los atributos temporales especificados en un dict.
    ...
    Argumentos:
        filtro (dict): Diccionario con dtypes key/str, val/list (e.g: {'mes':[6,12]})

    Devuelve:
      pandas.dataframe : df filtrado por mes o fecha.
    """


    for k,v in filtro.items():
        if len(v) > 1:
            print('Filtrando df por {}, entre {} y {}'.format(k,v[0],v[1]))
            df1 = df[(df[k] >= v[0]) & (df[k] <= v[1])]
            return df1

        elif len(v) == 1:
            print('Filtrando df para {} {}'.format(k,v[0]))
            df1 = df[(df[k] == v[0])]
            return df1

        else:
            print('No se aplica ningún filtro')

# 3. Integramos ambos pasos en una misma funcón
def compila_data(dict_input,key, filtro=None):
    '''
    Asigna atributos temporales a un df y filtra casos
    en función de los mismos.
    ...
    Argumentos:
    dict_input (dict): diccionario de dataframes con viajes por año
    key (str): año (e.g. '18')
    filtro (dict): diccionario con dtypes key/str, val/list (e.g: {'mes':[6,12]})

    Devuelve:
      pandas.dataframe : df de viajes origen/destino para un año determinado.
    '''

    df = atributos_temporales(dict_input, key)

    if filtro:
        df_filt = construye_filtros(df,filtro)
        return df_filt
    else:
        return df

In [ ]:
%%time
t1 = compila_data(dict_input=df, key='18', filtro={'mes':[12]}) # Ejemplo de uso: Filtra Diciembre 2018

In [ ]:
%%time
t2 = compila_data(dict_input=df, key='18', filtro={'mes':[6,12]}) # Ejemplo de uso: Filtra Junio:Diciembre 2018

In [ ]:
%%time
t3 = compila_data(dict_input=df, key='18',filtro={'fecha':['2018-12-31']}) # Ejemplo de uso: Filtra fecha

In [ ]:
%%time
# acá filtramos entre los días 24 y 31 del mes de diciembre
t4 = compila_data(dict_input=df, key='18',filtro={'fecha':['2018-12-24','2018-12-31']})

In [ ]:
#@title 4.3. Calculamos el grado de los nodos de nuestra red por hora usando **t1**

%%time

def crear_grafo_viajes(df_viajes):
    """
    Crea un MultiDiGraph a partir de un DataFrame de viajes en bicicleta.

    Args:
        df_viajes (pd.DataFrame): DataFrame que contiene los datos de viajes.

    Returns:
        nx.MultiDiGraph: Grafo dirigido que representa los viajes.
    """
    # Convertir las columnas de fechas
    df_viajes['bici_Fecha_hora_retiro'] = pd.to_datetime(df_viajes['bici_Fecha_hora_retiro'])
    df_viajes['hora'] = df_viajes['bici_Fecha_hora_retiro'].dt.hour

    # Crear el MultiDiGraph
    G = nx.MultiDiGraph()

    # Agregar nodos al grafo (de origen y destino) y sus atributos
    nodos_origen = df_viajes[['bici_nombre_estacion_origen']].drop_duplicates().rename(columns={'bici_nombre_estacion_origen': 'nombre'})
    nodos_destino = df_viajes[['bici_nombre_estacion_destino']].drop_duplicates().rename(columns={'bici_nombre_estacion_destino': 'nombre'})
    nodos = pd.concat([nodos_origen, nodos_destino]).drop_duplicates()

    # Crear un diccionario de nodos y sus atributos
    nodos_dict = nodos.set_index('nombre').to_dict('index')

    # Agregar nodos al grafo
    G.add_nodes_from(nodos_dict)

    # Crear las aristas con sus atributos
    aristas = df_viajes.apply(lambda row: (row['bici_nombre_estacion_origen'],
                                           row['bici_nombre_estacion_destino'],
                                           {'hora': row['hora']}), axis=1).tolist()

    # Agregar aristas al grafo
    G.add_edges_from(aristas)

    return G

# Función para calcular métricas de grado por hora
def ndg_by_hour(grafo):
    horas = range(24)
    data = []

    for hora in horas:
        subgrafo = grafo.edge_subgraph([(u, v, k) for u, v, k, d in grafo.edges(keys=True, data=True) if d['hora'] == hora])

        ndg = subgrafo.degree()
        idg = subgrafo.in_degree()
        odg = subgrafo.out_degree()

        estacion = [k for k, v in ndg]
        in_degree = [v for k, v in idg]
        out_degree = [v for k, v in odg]
        node_degree = [v for k, v in ndg]

        df_hora = pd.DataFrame(estacion, columns=['estacion'])
        df_hora['in_degree'] = in_degree
        df_hora['out_degree'] = out_degree
        df_hora['node_degree'] = node_degree
        df_hora['hora'] = hora

        data.append(df_hora)

    result_df = pd.concat(data)
    return result_df

G = crear_grafo_viajes(df_viajes=t1)
degree_df = ndg_by_hour(G)
merged_df = pd.merge(degree_df, estaciones, left_on='estacion', right_on='NOMBRE', how='left')
degree_gdf = gpd.GeoDataFrame(merged_df, geometry='geometry')

In [ ]:
#@title 4.4. Visualizamos el grado del nodo por hora

def mapa_estaciones(gdf, metrica, fecha):
    '''
    Plotea la cantidad de retiros de bicicletas por estación y hora
    para un período de tiempo determinado.
    ...
    Argumentos:
    gdf(gdf): GeoDataFrame.
    metrica (str): in_degree, out_degree o node_degree
    fecha(str): Título del plot.

    Devuelve:
      geopandas.geodataframe : gdf de retiros por estación y hora.
    '''

    fig = px.scatter_mapbox(gdf,
                            lat=gdf.geometry.y,lon=gdf.geometry.x,
                            hover_name="estacion",
                            animation_frame="hora",
                            size=metrica,
                            color_discrete_sequence=['#F05E23'],
                            opacity=0.9,
                            zoom=11,
                            height=600)

    fig.update_layout(mapbox_style="carto-darkmatter",
                      title_text = 'Usuarios/Retiros de bicicletas por estacion<br>({})'.format(fecha),
                      showlegend = True)

    fig.update_layout(margin={"r":1,"t":75,"l":0,"b":0})
    fig.show()

In [ ]:
mapa_estaciones(gdf=degree_gdf.to_crs(4326), metrica='out_degree', fecha='Diciembre 2018')